# Metabolite GPrediction Colab Workflow

This notebook clones the repository, installs dependencies, trains a SELFIES model with MLflow logging, evaluates generation quality, and runs sample inference without manual shell command editing.

In [ ]:
!git clone https://github.com/Gtedget/Metabolite_gprediction.git
%cd /content/Metabolite_gprediction
!pip install -r requirements.txt

In [ ]:
from datetime import datetime
from pathlib import Path

USE_GOOGLE_DRIVE_FOR_MLFLOW = False
if USE_GOOGLE_DRIVE_FOR_MLFLOW:
    from google.colab import drive
    drive.mount('/content/drive')
    MLFLOW_URI = 'file:/content/drive/MyDrive/metabolite_mlruns'
else:
    MLFLOW_URI = 'file:/content/mlruns'

RUN_NAME = datetime.utcnow().strftime('colab_%Y%m%d_%H%M%S')
ARTIFACT_DIR = Path('artifacts') / RUN_NAME
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print('MLflow URI:', MLFLOW_URI)
print('Run name:', RUN_NAME)
print('Artifact dir:', ARTIFACT_DIR)

In [ ]:
import subprocess

train_cmd = [
    'python', 'train.py',
    '--data', 'train.csv',
    '--epochs', '20',
    '--batch_size', '16',
    '--lr', '1e-4',
    '--device', 'cuda',
    '--representation', 'selfies',
    '--output_dir', 'artifacts',
    '--run_name', RUN_NAME,
    '--use_mlflow',
    '--mlflow_experiment', 'metabolite-predictor-colab',
    '--mlflow_run_name', RUN_NAME,
    '--mlflow_tracking_uri', MLFLOW_URI,
]
subprocess.run(train_cmd, check=True)
MODEL_PATH = str(ARTIFACT_DIR / 'trained_model.best.pt')
METADATA_PATH = str(ARTIFACT_DIR / 'trained_model.metadata.json')
print('Best checkpoint:', MODEL_PATH)
print('Metadata:', METADATA_PATH)

In [ ]:
eval_cmd = [
    'python', 'evaluate_generation.py',
    '--data', 'test.csv',
    '--model', MODEL_PATH,
    '--metadata', METADATA_PATH,
    '--top_k', '5',
    '--beam_width', '5',
    '--device', 'cuda',
    '--out', str(ARTIFACT_DIR / 'generation_eval.json'),
    '--use_mlflow',
    '--mlflow_experiment', 'metabolite-generation-eval-colab',
    '--mlflow_run_name', f'{RUN_NAME}-eval',
    '--mlflow_tracking_uri', MLFLOW_URI,
]
subprocess.run(eval_cmd, check=True)

## Inference and SoM View

This section runs inference on a sample precursor, shows ranked metabolite candidates, reports the top predicted transformation classes, and renders the highest-scoring SoM-like atoms from the reaction-center head.

In [ ]:
import json

from IPython.display import SVG, display
import pandas as pd
import torch

from inference import (
    _resolve_device,
    beam_search_candidates,
    load_model,
    predict_sites_of_metabolism,
    predict_top_transformations,
    render_sites_of_metabolism_svg,
 )

device = _resolve_device('cuda' if torch.cuda.is_available() else 'cpu')
model, tokenizer, metadata = load_model(MODEL_PATH, METADATA_PATH, device)

precursor_smiles = 'CC(=O)Oc1ccccc1C(=O)O'
output_json_path = ARTIFACT_DIR / 'inference.json'
output_svg_path = ARTIFACT_DIR / 'som.svg'

candidates, graph = beam_search_candidates(
    model,
    tokenizer,
    precursor_smiles,
    device,
    metadata=metadata,
    num_candidates=5,
    beam_width=5,
    max_len=tokenizer.max_len,
 )
top_transformations = predict_top_transformations(model, metadata, graph, top_k=5)
som_predictions, _ = predict_sites_of_metabolism(
    model,
    precursor_smiles,
    device,
    top_k=5,
    threshold=0.5,
 )
render_sites_of_metabolism_svg(
    precursor_smiles,
    som_predictions,
    str(output_svg_path),
    threshold=0.5,
 )

prediction_payload = {
    'precursor': precursor_smiles,
    'metabolite_candidates': candidates,
    'transformations': top_transformations,
    'sites_of_metabolism': som_predictions,
    'som_threshold': 0.5,
}
with open(output_json_path, 'w', encoding='utf-8') as f:
    json.dump(prediction_payload, f, indent=2)

print('Device:', device)
print('Precursor:', precursor_smiles)
print('Inference JSON:', output_json_path)
print('SoM SVG:', output_svg_path)

display(pd.DataFrame(candidates))
display(pd.DataFrame(top_transformations))
display(pd.DataFrame(som_predictions))
display(SVG(filename=str(output_svg_path)))